# fewshot-msme-inspect -- reproduce a result in Colab

Zero setup: run the cells top to bottom on a **CPU runtime** (the default; no GPU
needed, which is the point of the project). In about 10 minutes this

1. clones the repo and installs the pinned CPU environment,
2. downloads one MVTec AD category (`bottle`, ~150 MB),
3. runs the frozen CLIP + prototype-memory benchmark on it at 1/5/10 shots over
   the three committed seeds,
4. prints the result next to the number committed in the repo.

Numbers match the paper to within floating-point noise; latency will not, since
Colab's CPU is not the benchmark laptop.

In [ ]:
import os, subprocess
REPO = "https://github.com/Mounika-Reddy-0802/fewshot-msme-inspect.git"
# while the repo is private, put a read-only token in Colab Secrets as GITHUB_TOKEN
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = REPO.replace("https://", f"https://{token}@") if token else REPO
if not os.path.isdir("fewshot-msme-inspect"):
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
    subprocess.run(["git", "-C", "fewshot-msme-inspect", "remote", "set-url", "origin", REPO], check=True)
os.chdir("fewshot-msme-inspect")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -q open_clip_torch==3.3.0 onnx==1.22.0 onnxruntime==1.29.0 scikit-learn==1.6.1 pyyaml==6.0.3 psutil fvcore

In [ ]:
os.environ["FEWSHOT_DATA_ROOT"] = "/content/data"
os.environ["FEWSHOT_RESULTS_ROOT"] = "/content/results"
!python scripts/download_mvtec_category.py --category bottle

In [ ]:
!python scripts/run_benchmark.py --categories bottle --out /content/results/bottle.csv

In [ ]:
import csv, statistics
rows = list(csv.DictReader(open("/content/results/bottle.csv")))
committed = {1: 0.985, 5: 0.992, 10: 0.994}   # docs/week2_results.md, section 1
for k in (1, 5, 10):
    vals = [float(r["image_auroc"]) for r in rows if int(r["shots"]) == k]
    print(f"{k:>2}-shot  image AUROC {statistics.fmean(vals):.3f}  (committed {committed[k]:.3f}, "
          f"{len(vals)} seeds)")

## Going further

* `python scripts/reproduce.py --list` names the command for every table and figure.
* `python scripts/download_mvtec_category.py --all` fetches all 15 categories
  (~5 GB) for the full grid.
* The demo (`streamlit run app/demo.py`) is meant to run locally, not in Colab.